In [11]:
#5.aid_url_mapping.py
import pandas as pd
from pathlib import Path
import os, sys
import config
from urllib.parse import urlparse

# ---------- NAMING & PATHS ----------
FANDOM = urlparse(config.BASE_URL).netloc.split(".")[0]        # e.g., "alldimensions"
RAW_DATA_DIR = Path(config.FANDOM_DATA_DIR)                    # e.g., .../raw_data/alldimensions_fandom_data
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = RAW_DATA_DIR / f"master_spans_{FANDOM}.csv"
BASE_OUT = RAW_DATA_DIR / f"aid_url_mapping_{FANDOM}.csv"
A_OUT = RAW_DATA_DIR / f"master_csv_{FANDOM}.csv"
UNMATCHED_OUT = RAW_DATA_DIR / f"unmatched_{FANDOM}.csv"

In [12]:
# ---------- LOGIC ----------
def build_base_mapping():
    df = pd.read_csv(MASTER_PATH)
    base_df = df[["article_id", "page_url"]].drop_duplicates()
    base_df.to_csv(BASE_OUT, index=False)
    df, base_df = build_base_mapping()
    print(base_df.head())

In [13]:
# ---------- LOGIC ----------
def build_base_mapping():
    df = pd.read_csv(MASTER_PATH)
    base_df = df[["article_id", "page_url"]].drop_duplicates()
    base_df.to_csv(BASE_OUT, index=False)
    print(f"✅ Saved base mapping: {BASE_OUT} ({len(base_df)} rows)")
    return df, base_df

def match_articles(df_master, base_df):
    lookup = dict(zip(base_df["page_url"], base_df["article_id"]))
    A = df_master.copy()
    A["article_id_of_internal_link"] = A["resolved_url"].map(lookup)

    matched = int(A["article_id_of_internal_link"].notna().sum())
    unmatched = int(A["article_id_of_internal_link"].isna().sum())
    print("Matched:", matched)
    print("Unmatched:", unmatched)

    A.to_csv(A_OUT, index=False)
    A[A["article_id_of_internal_link"].isna()].to_csv(UNMATCHED_OUT, index=False)
    print(f"✅ Outputs saved: {A_OUT}, {UNMATCHED_OUT}")

# ---------- MAIN ----------
if __name__ == "__main__":
    df_master, base_df = build_base_mapping()
    match_articles(df_master, base_df)

✅ Saved base mapping: /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/aid_url_mapping_alldimensions.csv (3490 rows)
Matched: 0
Unmatched: 30424
✅ Outputs saved: /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/master_csv_alldimensions.csv, /home/sundeep/Fandom-Span-Identification-and-Retrieval/1.Fandom_Dataset_Collection/raw_data/alldimensions_fandom_data/unmatched_alldimensions.csv


In [14]:
print("page_url sample:", base_df["page_url"].head(5).to_list())
print("resolved_url sample:", df_master["resolved_url"].head(5).to_list())

page_url sample: ['https://alldimensions.fandom.com/wiki/All_dimensions_Wiki/wiki/-', 'https://alldimensions.fandom.com/wiki/All_dimensions_Wiki/wiki/-Flaws_in_the_World-', 'https://alldimensions.fandom.com/wiki/All_dimensions_Wiki/wiki/-One_Who_Stands_Before_God-', 'https://alldimensions.fandom.com/wiki/All_dimensions_Wiki/wiki/-_1', 'https://alldimensions.fandom.com/wiki/All_dimensions_Wiki/wiki/-_2']
resolved_url sample: ['https://static.wikia.nocookie.net/alldimensions/images/9/90/-.png/revision/latest?cb=20220110224816', 'https://alldimensions.fandom.com/wiki/(Tiny)', 'https://alldimensions.fandom.com/wiki/Axion', 'https://alldimensions.fandom.com/wiki/(Tiny)', 'https://alldimensions.fandom.com/wiki/The_amount_of_love_EA_receives']
